# QueryChat Interaction Log Notebook

This notebook explains why the interaction log exists, walks through a full data pipeline, and illustrates each step with concrete examples drawn from the EPL Performance Dashboard.

**Log columns**
| Column | Description |
|---|---|
| `Timestamp` | UTC time the query was submitted |
| `Query` | Natural-language text the user typed |
| `Response` | Summary string — row count + column list returned by the AI filter |

## Why We Need the Log

The **AI Explorer** tab lets users ask free-text questions such as:

> *"show Liverpool away wins"*  
> *"matches where the home team scored at least 4 goals"*  
> *"only draws in the 2021/22 season"*

Claude Haiku translates these into SQL, filters `df_all`, and renders charts.  
Without a log we have **zero visibility** into:

- What questions users actually ask (vs. what we designed for)
- Which queries silently return 0 rows (broken SQL generation)
- Which teams / seasons users care about most
- Whether AI filter usage grows or stalls after launch
- Edge-cases that should become new example prompts in the sidebar

The log is the **feedback loop** between user behaviour and product improvement.

```
User types query
      │
      ▼
QueryChat (Claude Haiku) → SQL → filters df_all
      │
      ▼
log_interaction(query, response_summary, timestamp)
      │
      ├─► Google Sheets  (shared, collaborative)
      └─► logs/querychat_log.csv  (local fallback)
              │
              ▼
         This notebook
              │   
              ▼           
        Usage analytics 
```

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from collections import Counter
import re as _re
print('Libraries loaded.')

## Load the Log

The app writes to `logs/querychat_log.csv` (or Google Sheets).  

In [ ]:
LOG_PATH = Path('logs/querychat_log.csv')

if LOG_PATH.exists():
    df_log = pd.read_csv(LOG_PATH, parse_dates=['Timestamp'])
    df_log.columns = df_log.columns.str.strip().str.capitalize()
    print(f'Loaded real log: {len(df_log)} rows from {LOG_PATH}')
else:
    df_log = pd.DataFrame(columns=['Timestamp', 'Query', 'Response'])
    print(f'No log file found at {LOG_PATH} — continuing with an empty log DataFrame.')

df_log['Timestamp'] = pd.to_datetime(df_log['Timestamp'], errors='coerce')
df_log.head()

### What questions users actually ask (vs. what we designed for)

In [ ]:
stopwords = set(['the','and','in','of','to','for','a','an','on','at','is','are','only','show','matches','where'])
if len(df_log) == 0:
    print('Log is empty — no user queries to summarise.')
else:
    queries = df_log['Query'].astype(str).str.strip()
    top_queries = queries.value_counts().head(10)
    print('Top user queries (top 10):')
    for q,c in top_queries.items():
        print(f'  {c:3d}x  {q}')

    token_lists = []
    for q in queries.fillna(''):
        toks = [t for t in _re.split(r'\W+', q.lower()) if len(t) > 2 and t not in stopwords and not t.isdigit()]
        token_lists.append(toks)
    
    uni = Counter(t for toks in token_lists for t in toks)
    bi = Counter((toks[i], toks[i+1]) for toks in token_lists for i in range(len(toks)-1))
    print('\nTop tokens (likely topics/teams):')
    for t,c in uni.most_common(10):
        print(f'  {t}: {c}')
    print('\nTop bigrams (common phrases):')
    for (a,b),c in bi.most_common(10):
        print(f'  {a} {b}: {c}')

### Which queries silently return 0 rows (broken SQL generation)

In [ ]:
def extract_row_count(response: str):
    m = _re.search(r'returned (\d+) rows', str(response))
    return int(m.group(1)) if m else None

if 'row_count' not in df_log.columns:
    df_log['row_count'] = df_log['Response'].apply(extract_row_count)
zeros = df_log[df_log['row_count'] == 0].copy()
print(f'Zero-result queries: {len(zeros)}')
if len(zeros):
    for _, r in zeros.iterrows():
        ts = r['Timestamp'] if pd.notna(r['Timestamp']) else ''
        print('  [{0}] {1}'.format(ts, r['Query']))
else:
    print('  None')

### Which teams / seasons users care about most

In [ ]:
df_log['mentioned_season'] = df_log['Query'].astype(str).str.extract(r'(\d{4}[/\-]\d{2})', expand=False)
season_counts = df_log['mentioned_season'].value_counts().head(10)
print('Top mentioned seasons:')
if len(season_counts):
    for s,c in season_counts.items():
        print(f'  {s}: {c}')
else:
    print('  None')


tokens = []
for q in df_log['Query'].astype(str).fillna(''):
    toks = [t for t in _re.split(r'\W+', q) if len(t) > 2 and not t.isdigit()]
    tokens.extend([t.lower() for t in toks])
from collections import Counter as _Counter
tok_counts = _Counter(tokens)
print('\nTop tokens (possible team names or topics):')
for t,c in tok_counts.most_common(10):
    print(f'  {t}: {c}')

print('\nNote: tokens are extracted from queries only; map them to known teams if desired.')